# Parte 0 - Tres fallas que no fallan

**Nombres: Jessica Ballesteros, Miguel Álvarez, Darlyn Ludeña**

## 0.a — El PDF que se indexa vacío (3 %)

**`ejemplos/instructivo_escaneado.pdf` no tiene capa de texto. Ingesta la carpeta `ejemplos/` con `python rag_pipeline.py` y pega la salida de la ingesta. Después, en `ingestion.py`, explica en dos frases qué comprueba `load_corpus` para no indexarlo y qué produciría sin esa comprobación con ese mismo archivo: un fragmento hecho de `[page=1] [page=2]`, sin una palabra del documento y sin ninguna excepción.**

In [1]:
from pathlib import Path
import os
import subprocess
import sys
import subprocess
import csv
import pandas as pd
from dotenv import load_dotenv

In [2]:
# Raíz del proyecto: se busca hacia arriba la carpeta que tiene pyproject.toml y rag_pipeline.py,
# así el notebook funciona igual lo abras desde notebooks/ o desde la raíz.
raiz = Path.cwd()
while raiz != raiz.parent and not ((raiz / "pyproject.toml").exists() and (raiz / "rag_pipeline.py").exists()):
    raiz = raiz.parent

sys.path.insert(0, str(raiz))
os.chdir(raiz)   # las rutas relativas del Lab (ejemplos/, corpus/) se resuelven desde la raíz

print("Raíz del proyecto:", raiz)

Raíz del proyecto: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña


In [3]:
ejemplos = raiz / "ejemplos"
pdf_escaneado = ejemplos / "instructivo_escaneado.pdf"

In [4]:
print("Directorio del proyecto:", raiz)
print("Carpeta ejemplos:", ejemplos)
print("PDF escaneado:", pdf_escaneado)

print("\nComprobaciones:")
print("ingestion.py:", (raiz / "ingestion.py").is_file())
print("rag_pipeline.py:", (raiz / "rag_pipeline.py").is_file())
print("carpeta ejemplos:", ejemplos.is_dir())
print("PDF escaneado:", pdf_escaneado.is_file())

Directorio del proyecto: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña
Carpeta ejemplos: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/ejemplos
PDF escaneado: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/ejemplos/instructivo_escaneado.pdf

Comprobaciones:
ingestion.py: True
rag_pipeline.py: True
carpeta ejemplos: True
PDF escaneado: True


Se procede a ejecutar el archivo `rag_pipeline.py`.

In [5]:
# Parte 0: sin clave, sin VPN y sin Docker (enunciado).
os.environ["EMBEDDING_BACKEND"] = "local"     # MiniLM, tope 128 tokens
os.environ["QDRANT_URL"] = ":memory:"
os.environ["CORPUS_DIR"] = "ejemplos"
# Se vacían las claves: rag_pipeline.py, al final, llama al generador y con una clave
# inválida en .env el proceso terminaría con un error de autenticación (y la salida
# guardada podría incluir parte de la clave). Aquí solo interesa la ingesta.
os.environ["OPENAI_API_KEY"] = ""
os.environ["ANTHROPIC_API_KEY"] = ""

print("EMBEDDING_BACKEND =", os.environ["EMBEDDING_BACKEND"])
print("QDRANT_URL        =", os.environ["QDRANT_URL"])
print("CORPUS_DIR        =", os.environ["CORPUS_DIR"])

EMBEDDING_BACKEND = local
QDRANT_URL        = :memory:
CORPUS_DIR        = ejemplos


In [6]:
# Se ejecuta el script tal como lo haría un compañero en la terminal, desde la raíz del
# proyecto (cwd=raiz). Antes corría desde notebooks/, CORPUS_DIR=ejemplos no existía ahí
# y la salida guardada era "ningún fragmento indexable", no la ingesta.
resultado = subprocess.run(
    [sys.executable, "rag_pipeline.py", "¿cuántos días de vacaciones puedo transferir?"],
    cwd=raiz,
    env=os.environ.copy(),
    capture_output=True,
    text=True,
)

print(resultado.stdout)
if resultado.stderr:
    print("STDERR:")
    print(resultado.stderr)
print("Código de retorno:", resultado.returncode)

assert "instructivo_escaneado.pdf" in resultado.stdout + resultado.stderr, \
    "La salida no menciona el PDF escaneado: revisa que se ejecutó sobre ejemplos/"

embeddings: local · sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 · tope 128 tokens · Qdrant :memory:
ingesta: nimbus_gastos.md: 1 página(s), 294 caracteres útiles → 1 fragmento(s)
ingesta: nimbus_remoto.md: 1 página(s), 296 caracteres útiles → 1 fragmento(s)
ingesta: nimbus_vacaciones.md: 1 página(s), 363 caracteres útiles → 1 fragmento(s)
indexados 3 fragmentos de 3 documento(s)
  0.681  nimbus_vacaciones.md / nimbus_vacaciones-0000  # Política de vacaciones de NimbusSoft Los empleados a tiempo completo acumulan …
  0.502  nimbus_remoto.md / nimbus_remoto-0000  # Política de trabajo remoto de NimbusSoft El trabajo remoto está permitido hast…
  0.485  nimbus_gastos.md / nimbus_gastos-0000  # Política de reembolso de gastos de NimbusSoft Los gastos de viaje se reembolsa…

[modo inspección (sin clave ni Ollama)] abstuvo=None
[modo inspección: no hay clave de API ni Ollama; este es el prompt]
Responde usando solo el contexto recuperado.
Si el contexto no contiene la respues

In [7]:
resultado.stderr

'Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.\n\nLoading weights:   0%|          | 0/199 [00:00<?, ?it/s]\nLoading weights: 100%|██████████| 199/199 [00:00<00:00, 8754.54it/s]\nAVISO ingesta: instructivo_escaneado.pdf: 0 caracteres útiles en 2 página(s). ¿Escaneado o fotografiado? NO se indexa: sin texto no hay nada que recuperar (OCR aparte).\n'

In [8]:
# Crear carpeta de resultados
carpeta_resultados =  raiz / "resultados" / "parte0"
carpeta_resultados.mkdir(parents=True, exist_ok=True)

# Guardar stdout en CSV
archivo_csv = carpeta_resultados / "salida_0a.csv"

with open(archivo_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["salida_stdout"])
    writer.writerow([resultado.stdout])
    writer.writerow(["salida_stderr"])
    writer.writerow([resultado.stderr])

print(f"\nSalida guardada en: {archivo_csv}")


Salida guardada en: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/resultados/parte0/salida_0a.csv


La salida cruda queda en `resultados/parte0/salida_0a.csv`. La línea que importa es el aviso:
`AVISO ingesta: instructivo_escaneado.pdf: 0 caracteres útiles en 2 página(s) … NO se indexa`.

**Las dos frases (también al final de `ingestion.py`):**

1. `load_corpus` cuenta, para cada documento, los caracteres útiles —letras y dígitos, sin las marcas `[page=N]` que añade la propia ingesta— y, si no llegan a `MIN_CARACTERES_UTILES` (200), lo anuncia en pantalla y no lo indexa; `instructivo_escaneado.pdf` tiene 0 caracteres útiles en 2 páginas porque es una imagen sin capa de texto.
2. Sin esa comprobación, `pypdf` devolvería texto vacío en cada página y la ingesta produciría un único fragmento hecho solo de `[page=1] [page=2]`, que se vectorizaría e indexaría sin ninguna excepción: el índice diría que el documento está, pero ninguna de sus palabras sería recuperable.

## 0.b El fragmento que se corta a 128 

**Con el MiniLM de la Parte 0 (EMBEDDING_BACKEND=local, tope de 128 tokens), ingesta ejemplos/ pidiendo chunk_tokens=900 y pega el aviso. Después toma un texto de unas 900 palabras, cuenta sus tokens con el tokenizador del modelo, y compara el vector del texto entero con el del texto recortado a 
max_seq_length tokens: el coseno es ≈ 1. Todo lo que no cupo nunca llegó al índice. Es el ejercicio 5.1 del lunes, sobre el laboratorio.**

In [9]:
from pathlib import Path
from rag_pipeline import RagPipeline
import os
import numpy as np

In [10]:


os.environ["EMBEDDING_BACKEND"] = "local"
os.environ["QDRANT_URL"] = ":memory:"
os.environ["CORPUS_DIR"] = "ejemplos"

print("EMBEDDING_BACKEND =", os.environ["EMBEDDING_BACKEND"])
print("QDRANT_URL        =", os.environ["QDRANT_URL"])

EMBEDDING_BACKEND = local
QDRANT_URL        = :memory:


In [11]:
pipeline = RagPipeline()

chunks_900 = pipeline.ingest(
    raiz / "ejemplos",        # ruta absoluta: antes, Path("ejemplos") desde notebooks/ daba 0 fragmentos
    chunk_tokens=900
)

print(f"\nFragmentos obtenidos: {len(chunks_900)}")
print(f"Tope del modelo: {pipeline.max_seq_tokens} tokens")

/Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 26253.15it/s]


ingesta: nimbus_gastos.md: 1 página(s), 294 caracteres útiles → 1 fragmento(s)
ingesta: nimbus_remoto.md: 1 página(s), 296 caracteres útiles → 1 fragmento(s)
ingesta: nimbus_vacaciones.md: 1 página(s), 363 caracteres útiles → 1 fragmento(s)

Fragmentos obtenidos: 3
Tope del modelo: 128 tokens


AVISO ingesta: chunk_tokens=900 supera el tope del modelo (128 tokens): lo que pase de 126 tokens no llega al índice —el MiniLM lo trunca en silencio; la H200 rechaza la petición—.
AVISO ingesta: instructivo_escaneado.pdf: 0 caracteres útiles en 2 página(s). ¿Escaneado o fotografiado? NO se indexa: sin texto no hay nada que recuperar (OCR aparte).


In [12]:
faqs = [
    "Para resetear tu contraseña entra a Configuración > Seguridad y pulsa 'Olvidé mi clave'.",
    "Los reembolsos se procesan en 5 a 7 días hábiles tras aprobar la solicitud.",
    "Puedes exportar tus datos en formato CSV desde el panel de administración.",
    "La factura electrónica se emite automáticamente al confirmar el pago.",
    "Para cancelar tu suscripción ve a Cuenta > Plan y selecciona 'Cancelar'.",
    "El soporte técnico atiende de lunes a viernes de 9h a 18h por chat.",
    "La API permite hasta 100 solicitudes por minuto en el plan básico.",
    "Aceptamos tarjetas de crédito, débito y transferencias bancarias.",
]

# Texto de 900 palabras, como pide el enunciado (Ejercicio 5.1 del lunes):
# se repiten las FAQs y se corta exactamente en la palabra 900.
texto_largo = " ".join(((" ".join(faqs) + " ") * 10).split()[:900])

print("Palabras del texto:", len(texto_largo.split()))

Palabras del texto: 900


In [13]:
### se obtiene el tokenizador y el tope del modelo

modelo = pipeline.encoder._modelo

tokenizador = modelo.tokenizer
tope = modelo.max_seq_length

print("Modelo:", modelo)
print("max_seq_length:", tope)

Modelo: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
)
max_seq_length: 128


In [14]:
### se cuentan los tokens y se construye el recorte

ids = tokenizador(
    texto_largo,
    add_special_tokens=True,
    truncation=False
)["input_ids"]

# se conservan los tokens que caben dentro del límite del modelo.
recorte = tokenizador.decode(ids[1:tope - 1])

print("Tokens del texto:", len(ids))
print("Tope del modelo:", tope)
print("Tokens del recorte:", len(
    tokenizador(
        recorte,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]
))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1297 > 128). Running this sequence through the model will result in indexing errors


Tokens del texto: 1297
Tope del modelo: 128
Tokens del recorte: 128


In [15]:
### se comparan los embeddings


v_todo, v_recorte = pipeline.encoder.encode(
    [texto_largo, recorte]
)

coseno = float(v_todo @ v_recorte)

print(f"Tokens del texto: {len(ids)}")
print(f"Max_seq_length del modelo: {tope}")
print(f"Coseno(texto entero, recortado a {tope} tokens) = {coseno:.4f}")


Tokens del texto: 1297
Max_seq_length del modelo: 128
Coseno(texto entero, recortado a 128 tokens) = 1.0000


Se guardan los resultados del ejercicio dentro de la carpeta `resultados/parte0` incluidos los tokens y el coseno.

In [16]:
carpeta_resultados = raiz / "resultados" / "parte0"
carpeta_resultados.mkdir(parents=True, exist_ok=True)

archivo = carpeta_resultados / "salida_0b.csv"

with open(archivo, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)

    writer.writerow(["elemento", "valor"])
    writer.writerow(["tokens_del_texto", len(ids)])
    writer.writerow(["max_seq_length", tope])
    writer.writerow(["tokens_del_recorte", len(
        tokenizador(
            recorte,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )])
    writer.writerow(["coseno", coseno])
    writer.writerow(["chunk_tokens_prueba", 900])
    writer.writerow(["chunk_tokens_parte1", 512])

print(f"Resultados guardados en: {archivo}")

Resultados guardados en: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/resultados/parte0/salida_0b.csv


In [17]:
N = len(ids)
N

1297

In [18]:
coseno

1.0

### Conclusión 0.b

Los números salen de las celdas anteriores (y de `resultados/parte0/salida_0b.csv`): el texto de 900 palabras ocupa bastantes más tokens que el tope de 128 del MiniLM, y el coseno entre el vector del texto entero y el del texto recortado a `max_seq_length` es ≈ 1. El modelo solo "vio" los primeros 128 tokens: todo lo que venía después **nunca llegó al índice**, y la librería no avisa. El Lab-02 sí avisa (el `AVISO ingesta` de la celda de `chunk_tokens=900`).

**`chunk_tokens` para la Parte 1: 512 (con 102 de solapamiento, un quinto).** Que bge-m3 admita 8192 tokens dice qué *cabe*, no qué *sirve*. Nuestro corpus son papers en inglés: un párrafo ocupa del orden de 100–250 tokens, así que 512 tokens son 2–3 párrafos, suficiente para que una afirmación llegue con su contexto inmediato y lo bastante acotado para que el vector represente una idea. Un fragmento de 8192 tokens equivaldría a casi todo el paper de Vaswani (25 fragmentos de 512): su vector sería el promedio de muchos temas, cualquier pregunta puntual lo encontraría "medio parecido", y le entregaríamos al generador miles de tokens irrelevantes.

## 0.c El índice que no se queja

**Sobre 
ejemplos/ indexado, pregunta algo cuya respuesta no está (
"¿cuál es la política de mascotas?"
) y
pega los tres vecinos con sus puntajes. Tienen aspecto sano. Después pregunta algo cuya respuesta sí está en el
PDF escaneado que la 0.a no indexó (
"¿qué exige el instructivo de prácticas de campo sobre el seguro
de accidentes?"
) y pega los tres vecinos. También tienen aspecto sano.**

In [19]:
import os

os.environ["EMBEDDING_BACKEND"] = "local"
os.environ["QDRANT_URL"] = ":memory:"
os.environ["CORPUS_DIR"] = "ejemplos"

print("EMBEDDING_BACKEND =", os.environ["EMBEDDING_BACKEND"])
print("QDRANT_URL =", os.environ["QDRANT_URL"])
print("CORPUS_DIR =", os.environ["CORPUS_DIR"])

EMBEDDING_BACKEND = local
QDRANT_URL = :memory:
CORPUS_DIR = ejemplos


In [20]:
from pathlib import Path
from rag_pipeline import RagPipeline

pipeline = RagPipeline()

chunks = pipeline.ingest(
   raiz / "ejemplos"
)

print(f"Fragmentos obtenidos: {len(chunks)}")
print(f"Tope del modelo: {pipeline.max_seq_tokens} tokens")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19131.00it/s]


ingesta: nimbus_gastos.md: 1 página(s), 294 caracteres útiles → 1 fragmento(s)
ingesta: nimbus_remoto.md: 1 página(s), 296 caracteres útiles → 1 fragmento(s)
ingesta: nimbus_vacaciones.md: 1 página(s), 363 caracteres útiles → 1 fragmento(s)
Fragmentos obtenidos: 3
Tope del modelo: 128 tokens


AVISO ingesta: instructivo_escaneado.pdf: 0 caracteres útiles en 2 página(s). ¿Escaneado o fotografiado? NO se indexa: sin texto no hay nada que recuperar (OCR aparte).


In [21]:
## indexando a quadrant en memoria
pipeline.index(chunks)

print("Índice creado correctamente.")
print("Colección:", pipeline.collection)

Índice creado correctamente.
Colección: mmia6013_rag


In [22]:
pregunta_1 = "¿Cuál es la política de mascotas?"

resultados_1 = pipeline.retrieve(
    pregunta_1,
    top_k=3
)

print("PREGUNTA:")
print(pregunta_1)
print("\nTOP 3 VECINOS:\n")

for i, resultado in enumerate(resultados_1, start=1):
    print(f"--- Vecino {i} ---")
    print(f"Score:    {resultado['score']:.4f}")
    print(f"Documento: {resultado['document']}")
    print(f"Chunk ID:  {resultado['chunk_id']}")
    print(f"Texto:    {resultado['text']}")
    print()

PREGUNTA:
¿Cuál es la política de mascotas?

TOP 3 VECINOS:

--- Vecino 1 ---
Score:    0.0632
Documento: nimbus_gastos.md
Chunk ID:  nimbus_gastos-0000
Texto:    # Política de reembolso de gastos de NimbusSoft Los gastos de viaje se reembolsan presentando factura dentro de los 30 días posteriores al gasto. El límite diario de alimentación en viajes es de 45 USD. Los pasajes aéreos deben comprarse en clase económica salvo vuelos de más de 8 horas, donde se permite económica premium con aprobación del gerente de área.

--- Vecino 2 ---
Score:    0.0339
Documento: nimbus_remoto.md
Chunk ID:  nimbus_remoto-0000
Texto:    # Política de trabajo remoto de NimbusSoft El trabajo remoto está permitido hasta 3 días por semana para todos los roles excepto soporte de infraestructura on-site. Los días remotos se coordinan con el líder de equipo. Para trabajar desde el exterior del país se requiere aprobación de Recursos Humanos con 30 días de anticipación y un máximo de 60 días por año.

--- Vecino

In [23]:
# Crear una fila por cada vecino
datos_1 = []

for i, resultado in enumerate(resultados_1, start=1):
    datos_1.append({
        "pregunta": pregunta_1,
        "rank": i,
        "score": resultado["score"],
        "documento": resultado["document"],
        "chunk_id": resultado["chunk_id"],
        "texto": resultado["text"]
    })

# Crear DataFrame
df_0c_1 = pd.DataFrame(datos_1)

In [24]:
pregunta_2 = (
    "¿Qué exige el instructivo de prácticas de campo "
    "sobre el seguro de accidentes?"
)

resultados_2 = pipeline.retrieve(
    pregunta_2,
    top_k=3
)

print("PREGUNTA:")
print(pregunta_2)
print("\nTOP 3 VECINOS:\n")

for i, resultado in enumerate(resultados_2, start=1):
    print(f"--- Vecino {i} ---")
    print(f"Score:    {resultado['score']:.4f}")
    print(f"Documento: {resultado['document']}")
    print(f"Chunk ID:  {resultado['chunk_id']}")
    print(f"Texto:    {resultado['text']}")
    print()

PREGUNTA:
¿Qué exige el instructivo de prácticas de campo sobre el seguro de accidentes?

TOP 3 VECINOS:

--- Vecino 1 ---
Score:    0.2289
Documento: nimbus_remoto.md
Chunk ID:  nimbus_remoto-0000
Texto:    # Política de trabajo remoto de NimbusSoft El trabajo remoto está permitido hasta 3 días por semana para todos los roles excepto soporte de infraestructura on-site. Los días remotos se coordinan con el líder de equipo. Para trabajar desde el exterior del país se requiere aprobación de Recursos Humanos con 30 días de anticipación y un máximo de 60 días por año.

--- Vecino 2 ---
Score:    0.1679
Documento: nimbus_gastos.md
Chunk ID:  nimbus_gastos-0000
Texto:    # Política de reembolso de gastos de NimbusSoft Los gastos de viaje se reembolsan presentando factura dentro de los 30 días posteriores al gasto. El límite diario de alimentación en viajes es de 45 USD. Los pasajes aéreos deben comprarse en clase económica salvo vuelos de más de 8 horas, donde se permite económica premium co

In [25]:
# Crear una fila por cada vecino
datos_2 = []

for i, resultado in enumerate(resultados_2, start=1):
    datos_2.append({
        "pregunta": pregunta_2,
        "rank": i,
        "score": resultado["score"],
        "documento": resultado["document"],
        "chunk_id": resultado["chunk_id"],
        "texto": resultado["text"]
    })

# Crear DataFrame
df_0c_2 = pd.DataFrame(datos_2)


In [26]:
df_0c_2

,pregunta,rank,score,documento,chunk_id,texto
0,¿Qué exige el instructivo de prácticas de camp...,1,0.228869,nimbus_remoto.md,nimbus_remoto-0000,# Política de trabajo remoto de NimbusSoft El ...
1,¿Qué exige el instructivo de prácticas de camp...,2,0.167904,nimbus_gastos.md,nimbus_gastos-0000,# Política de reembolso de gastos de NimbusSof...
2,¿Qué exige el instructivo de prácticas de camp...,3,0.097349,nimbus_vacaciones.md,nimbus_vacaciones-0000,# Política de vacaciones de NimbusSoft Los emp...


In [27]:
df_0c = pd.concat(
    [df_0c_1, df_0c_2],
    ignore_index=True
)

display(df_0c)

,pregunta,rank,score,documento,chunk_id,texto
0,¿Cuál es la política de mascotas?,1,0.063235,nimbus_gastos.md,nimbus_gastos-0000,# Política de reembolso de gastos de NimbusSof...
1,¿Cuál es la política de mascotas?,2,0.033908,nimbus_remoto.md,nimbus_remoto-0000,# Política de trabajo remoto de NimbusSoft El ...
2,¿Cuál es la política de mascotas?,3,-0.028547,nimbus_vacaciones.md,nimbus_vacaciones-0000,# Política de vacaciones de NimbusSoft Los emp...
3,¿Qué exige el instructivo de prácticas de camp...,1,0.228869,nimbus_remoto.md,nimbus_remoto-0000,# Política de trabajo remoto de NimbusSoft El ...
4,¿Qué exige el instructivo de prácticas de camp...,2,0.167904,nimbus_gastos.md,nimbus_gastos-0000,# Política de reembolso de gastos de NimbusSof...
5,¿Qué exige el instructivo de prácticas de camp...,3,0.097349,nimbus_vacaciones.md,nimbus_vacaciones-0000,# Política de vacaciones de NimbusSoft Los emp...


In [28]:
carpeta_resultados = raiz / "resultados" / "parte0"
carpeta_resultados.mkdir(parents=True, exist_ok=True)

archivo = carpeta_resultados / "salida_0c.csv"

df_0c.to_csv(
    archivo,
    index=False,
    encoding="utf-8-sig"
)

print(f"Resultados guardados en: {archivo}")

Resultados guardados en: /Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/resultados/parte0/salida_0c.csv


### Conclusiones

1. **Los seis vecinos "tienen aspecto sano".** Qdrant siempre devuelve los k vecinos más cercanos, ordenados, aunque ninguno responda: no existe un "no encontré nada". Con el MiniLM los puntajes son bajos en términos absolutos, pero no sirven como umbral: la pregunta cuya respuesta **sí existe** (en el PDF escaneado) obtiene puntajes más altos (≈ 0,23) que la de mascotas (≈ 0,06), y con bge-m3 (Partes 1 y 2) el mejor vecino ronda 0,55–0,70 tanto cuando acierta como cuando falla.

2. **Por qué ningún Hit Rate puede detectarlo:** el Hit Rate solo se calcula cuando existe un documento fuente contra el cual comparar lo recuperado; en la primera pregunta ese documento no existe, y en la segunda no está en el índice (la 0.a no lo indexó), así que para el evaluador tampoco existe. En los dos casos la recuperación "funciona" y el fallo solo aparece en la respuesta: por eso el golden set lleva preguntas **negativas** y el evaluador mide si el sistema **se abstiene**.